<div style="text-align: center; font-weight: bold;">
    <h1>EHR Preprocessing 1: Data Cleaning</h1>
    <h4>Author: Vidul Ayakulangara Panickan</h3>
</div>



Cleaning EHR data involves multiple substeps that vary depending on the type of data modality and the intended downstream analyis.  
In this tutorial, we focus on cleaning the structured data from the MIMIC-IV *hosp* module by 

> 1. Assessing Missingness
> 2. Filtering Irrelevant Information
> 3. Standardizing Schema
> 4. Identifying Erronous Data
> 5. Removing Redundant Data

In [1]:
# Importing required libraries and setting up the workspace

import os
import sys
import time
import logging
import pandas as pd
from tqdm import tqdm
from IPython.display import clear_output
from IPython.display import display

# Set pandas options to expand all data within rows
pd.set_option('display.max_columns', None)      
pd.set_option('display.max_colwidth', None) 

#Update base directory to your EHR_TUTORIAL_WORKSPACE path
base_directory ="/n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE"

# Setting up Directory to save Cleaned Data
cleaned_rawdata_directory = os.path.join(base_directory, 'processed_data', 'step3_cleaned_rawdata')
os.makedirs(cleaned_rawdata_directory, exist_ok=True)

print(f"Directory created at: {cleaned_rawdata_directory}")

Directory created at: /n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE/processed_data/step3_cleaned_rawdata


### Cleaning Diagnoses as an Example

We will walk through the steps involved in cleaning, using the diagnosis data from MIMIC as an example.. Once these steps are outlined, we will encapsulate them into a reusable function which we will use on other mimic datasets (Procedure, Labs, Medication) required for this tutorial. First, ensure that you have assembled all the necessary data.  

In [2]:
# The diagnosis data is available in diagnoses_icd_file and the time of recording is available in admissions_file

diagnoses_icd_file = os.path.join(base_directory, "raw_data", "structured_data","physionet.org", "files", "mimiciv", "3.1", "hosp","diagnoses_icd.csv")
admissions_file  = os.path.join(base_directory, "raw_data", "structured_data","physionet.org", "files", "mimiciv", "3.1", "hosp","admissions.csv")

diagnoses_icd = pd.read_csv(diagnoses_icd_file, dtype=str)
admissions = pd.read_csv(admissions_file, dtype=str)

# Listing all columns
display(diagnoses_icd.columns)
display(admissions.columns)

Index(['subject_id', 'hadm_id', 'seq_num', 'icd_code', 'icd_version'], dtype='object')

Index(['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime',
       'admission_type', 'admit_provider_id', 'admission_location',
       'discharge_location', 'insurance', 'language', 'marital_status', 'race',
       'edregtime', 'edouttime', 'hospital_expire_flag'],
      dtype='object')

### Assessing Missingness

Now before combining the tables, we need to check for missing values in the raw EHR files. While working with large scale EHR data, rows with missing dates or ICD codes are typically removed if the proportion of missing values is small. However, if substantial records have missing values, further investigation is required before proceeding, as systematic missingness may indicate underlying data quality issues.

In [3]:
print("Diagnoses Table - Missing Values Count")
diagnoses_icd_missing_df = pd.DataFrame({'Column': diagnoses_icd.columns,'Missing_Values': diagnoses_icd.isna().sum()})
display(diagnoses_icd_missing_df)

print("Admissions Table - Missing Values Count")
admissions_missing_df = pd.DataFrame({'Column': admissions.columns,'Missing_Values': admissions.isna().sum()})
display(admissions_missing_df )

Diagnoses Table - Missing Values Count


,Column,Missing_Values
subject_id,subject_id,0
hadm_id,hadm_id,0
seq_num,seq_num,0
icd_code,icd_code,0
icd_version,icd_version,0


Admissions Table - Missing Values Count


,Column,Missing_Values
subject_id,subject_id,0
hadm_id,hadm_id,0
admittime,admittime,0
dischtime,dischtime,0
deathtime,deathtime,534238
admission_type,admission_type,0
admit_provider_id,admit_provider_id,4
admission_location,admission_location,1
discharge_location,discharge_location,149818
insurance,insurance,9355


### Filtering Irrelevant Information

Next, we will identify the variables required for the downstream task and retain only those. For this example, we require **subject_id**, **hadm_id**, and **icd_code** from the diagnoses table, and **subject_id**, **hadm_id**, and **admittime** from the admissions table to construct the timestamped diagnoses dataset. 
Since there are no missing values in these fields, we can proceed with joining the tables.

> **Note:** Not all datasets require the same set of variables.  
> For example, when working with laboratory data, it is important to retain the laboratory values and their associated units in addition to the laboratory codes.  The specific variables to be preserved will depend on the requirements of the downstream task. In this tutorial, we do not make use of laboratory values.



In [4]:
# Merging diagnoses_icd and admissions tables on 'subject_id' and 'hadm_id' columns

diagnoses_icd = pd.read_csv(diagnoses_icd_file, dtype=str)
admissions = pd.read_csv(admissions_file, dtype=str)
timed_diagnoses_icd = pd.merge(
    diagnoses_icd,
    admissions[["subject_id", "hadm_id", "admittime"]],
    how="left",
    on=["subject_id", "hadm_id"],
)
display(timed_diagnoses_icd.head())

,subject_id,hadm_id,seq_num,icd_code,icd_version,admittime
0,10000032,22595853,1,5723,9,2180-05-06 22:23:00
1,10000032,22595853,2,78959,9,2180-05-06 22:23:00
2,10000032,22595853,3,5715,9,2180-05-06 22:23:00
3,10000032,22595853,4,07070,9,2180-05-06 22:23:00
4,10000032,22595853,5,496,9,2180-05-06 22:23:00


After performing any merge operation, it's recommended to check whether the resulting table has any missing values. 

Note: If there are a significant number of missing values, you should investigate further to identify underlying cause. For example, the join columns might be of different data types (like 'subject_id' can be integer in one table and string in another table), which can cause the join operation to  fail.

In [5]:
admissions_missing_df = pd.DataFrame({'Column': timed_diagnoses_icd.columns,'Missing_Values': timed_diagnoses_icd.isna().sum()})
display(admissions_missing_df)

,Column,Missing_Values
subject_id,subject_id,0
hadm_id,hadm_id,0
seq_num,seq_num,0
icd_code,icd_code,0
icd_version,icd_version,0
admittime,admittime,0


In the previous *EHR Preprocessing: Getting Started* notebook, we introduced the key data elements: **Patient ID** (`subject_id`), **Event/Observation** (`icd_code`), and **Time** (`date`). Records missing any of these essential elements should be removed prior to further analysis.

In [6]:
# Check columns of interest. If you have records where dates, icd_code or subect_ids are null, remove them. 

print(f"Table shape before null records have been removed {timed_diagnoses_icd.shape}")

timed_diagnoses_icd.dropna(subset=['admittime', 'icd_code', 'icd_version', 'subject_id'], how="any", inplace=True)

print(f"Table shape after null records have been removed {timed_diagnoses_icd.shape}")

Table shape before null records have been removed (6364488, 6)
Table shape after null records have been removed (6364488, 6)


Next, we truncate the timestamp to retain only the date, since the hour/minute/second are not required for our analysis.  

> **Note:** When working with datasets time series medical data like ECGs, the full timestamp should be preserved.  
> For the purposes of this tutorial, the date alone is sufficient.



In [7]:
timed_diagnoses_icd["admittime"] = timed_diagnoses_icd["admittime"].str[:10]

display(timed_diagnoses_icd.head())

,subject_id,hadm_id,seq_num,icd_code,icd_version,admittime
0,10000032,22595853,1,5723,9,2180-05-06
1,10000032,22595853,2,78959,9,2180-05-06
2,10000032,22595853,3,5715,9,2180-05-06
3,10000032,22595853,4,07070,9,2180-05-06
4,10000032,22595853,5,496,9,2180-05-06


### Standardizing Schema


Cleaning a dataset also involves renaming and restructuring columns and tables to ensure that newly generated datasets remain consistent.   Here, we rename **admittime** to **date** to maintain consistency with other datasets that will be created later.  

> **Note:** This convention allows alignment across different data datasets such as medications, laboratory results, and procedures, which will also use **date** as the standard timestamp field.


In [8]:
timed_diagnoses_icd=timed_diagnoses_icd[['subject_id','icd_code','icd_version','admittime']]
timed_diagnoses_icd = timed_diagnoses_icd.rename(columns={'admittime': 'date'})
display(timed_diagnoses_icd.head(5))

,subject_id,icd_code,icd_version,date
0,10000032,5723,9,2180-05-06
1,10000032,78959,9,2180-05-06
2,10000032,5715,9,2180-05-06
3,10000032,07070,9,2180-05-06
4,10000032,496,9,2180-05-06


#### Identifying Erronous Data

Define a valid date range for the dataset (not applicable to MIMIC-IV).

It is important in real-world datasets to ensure that dates fall within a reasonable range.  For instance, records with dates prior to the 1980s or beyond the current year should be excluded. The code for this operation is provided below. For MIMIC-IV, however this step is not required since dates are adjusted and won't make sense. 


> **Note on laboratory data:**  
> Erroneous values and outliers are also frequently encountered in laboratory datasets. While impossible values  can be safely filtered out, other issues require more care.  Laboratory results may appear erroneous when the same test is recorded in different units (for example, hemoglobin reported in g/dL versus mmol/L).  In such cases, applying domain knowledge to normalize units is essential for ensuring consistency and preventing valid results from being misclassified as errors. For this tutorial, we do not use laboratory values, but these checks are critical when working with real world lab data.

In [10]:
# diagnoses_icd = diagnoses_icd[
#     (diagnoses_icd["date"].str[:4].astype(int) >= 1980)
#     & (diagnoses_icd["date"].str[:4].astype(int) <= 2024)
# ]

### Removing Redundant Data

Duplicate records are prevalent in EHR datasets and can arise in multiple forms.  
These include **exact duplicates**, where identical rows are repeated, and **semantic duplicates**, where different medical codes represent the same underlying clinical concept.  

The deduplication step is applied both **before and after code standardization** (described in EHR Preprocessing 2 Code Rollup Notebook) to ensure that all redundant information is removed.  This process is essential to prevent double counting and reducing data noise.

In [11]:
# Check for duplicated rows in your data

if timed_diagnoses_icd.duplicated().sum() > 0:
    
    initial_row_count=timed_diagnoses_icd.shape[0]
    
    print(f"Initial table size {initial_row_count}")
    print("Duplicate rows found. Removing duplicates :")
    
    timed_diagnoses_icd = timed_diagnoses_icd.drop_duplicates()  # Remove duplicate rows
    final_row_count=timed_diagnoses_icd.shape[0]
    
    print(f"Records deleted: {initial_row_count - final_row_count}")
    print(f"Table size after removing duplicates : {final_row_count}")

else:
    print("No duplicate rows found.")

No duplicate rows found.


## Defining Cleaning Functions 

Next, we will define the cleaning functions based on the code above, so they can be reused for other datasets

In [12]:
## Ignore the function below, we are defining functions so that we can print and save the info in a log file

import os
import sys
import time
import logging
import pandas as pd
from tqdm import tqdm
from IPython.display import clear_output
from IPython.display import display

# Set pandas options to expand all data within rows
pd.set_option('display.max_columns', None)      
pd.set_option('display.max_colwidth', None) 


def setup_logger(log_type,log_file):

    log_folder = os.path.join("log_folder", log_type)
    
    # Ensure the log folder exists
    os.makedirs(log_folder, exist_ok=True)
    
    # Define the full path for the log file
    log_filepath = os.path.join(log_folder, log_file)

    # Delete the log file if it exists
    if os.path.exists(log_filepath):
        os.remove(log_filepath)
    
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    logging.basicConfig(level=logging.DEBUG, format='%(message)s', handlers=[logging.FileHandler(log_filepath)])

# Define the function to print and log messages
def print_and_log_cleaning(message):
    logging.info(message)     



def missing_values_summary(df):
    missing_values = df.isna().sum()
    missing_df = pd.DataFrame({'Column Name': missing_values.index, 'Missing Values Count': missing_values.values})
    print_and_log_cleaning("Missing Values Count:")
    print_and_log_cleaning(missing_df)



def clean_data(df, cols_of_interest, time_col):

    missing_values_summary(df)

    print_and_log_cleaning(f"Initial number of rows: {df.shape[0]}")

    print_and_log_cleaning("Keeping only the essential columns")
    df = df[cols_of_interest]

    if df.isna().sum().any():
        df = df.dropna()
        print_and_log_cleaning(f"Number of rows after dropping na rows: {df.shape[0]}")
    else:
        print_and_log_cleaning("No rows with missing values to drop.")

    print_and_log_cleaning("Extracting only the date info")
    df[time_col] = df[time_col].str[:10]

    print_and_log_cleaning("Renaming the columns")
    df = df.rename(columns={time_col: "date"})

    print_and_log_cleaning("Checking for duplicate rows")
    if df.duplicated().sum() > 0:
        print_and_log_cleaning("Duplicate rows found. Removing duplicates:")
        df = df.drop_duplicates()
        return df
    else:
        print_and_log_cleaning("No duplicate rows found.")
        return df


def clean_data_batch_supportfunc(df, cols_of_interest):

    missing_values_summary(df)

    print_and_log_cleaning(f"Initial number of rows: {df.shape[0]}")

    if df.isna().sum().any():
        df.dropna(inplace=True)
        print_and_log_cleaning(f"Number of rows after dropping na rows: {df.shape[0]}")
    else:
        print_and_log_cleaning("No rows with missing values to drop.")

    print_and_log_cleaning("Extracting only the date info")
    df[cols_of_interest['date']] = df[cols_of_interest['date']].str[:10]
  

    print_and_log_cleaning("Renaming the columns")
    df = df.rename(columns={cols_of_interest['date']: "date"})
    df = df.rename(columns={cols_of_interest['code']: "code"})

    print_and_log_cleaning("Checking for duplicate rows")
    if df.duplicated().sum() > 0:
        print_and_log_cleaning("Duplicate rows found. Removing duplicates:")
        df = df.drop_duplicates()
        return df
    else:
        print_and_log_cleaning("No duplicate rows found.")
        return df


def file_line_count(file_path):
    count = 0
    with open(file_path, 'r') as file:
        for line in file:
            count += 1
    return count


def clean_data_by_batch(input_file_path, cleaned_output_dir ,patient_ids, cols_of_interest, data_name, num_rows_to_load=1500000):
    #Coding system will be a list of columns if the coding system is defined in the input data, if not, you can pass a single value list

    filename = os.path.splitext(os.path.basename(input_file_path))[0]

    # Set up logger for cleaning function
    setup_logger("cleaning",f"{data_name}_{filename}.txt")  # Create log folder if necessary and set up logging

 
    # Create a dictionary for batch_num lookup based on subject_id
    batch_lookup = dict(zip(patient_ids['subject_id'], patient_ids['batch_num']))
    
    # Get the list of unique batch numbers from patient_ids
    unique_batch_nums = patient_ids['batch_num'].unique()

    output_dir = f'{cleaned_output_dir}/{data_name}'
    os.makedirs(output_dir, exist_ok=True)

    # Iterate over each batch number
    for batch_num in tqdm(unique_batch_nums, desc="Processing batches", unit="batch"):
        
        print_and_log_cleaning(f"\nProcessing batch {batch_num}:")

        collected_data = []

        non_none_cols_of_interest = [item for key, item in cols_of_interest.items() if item is not None and key != "coding_system"
]
        
        chunk_iter = pd.read_csv(input_file_path, chunksize=num_rows_to_load, usecols=non_none_cols_of_interest , dtype=str)
        
        # Process each chunk and collect data for the current batch
        for chunk_idx, chunk in enumerate(chunk_iter):
            print_and_log_cleaning(f"------ Processing  batch {batch_num} chunk{chunk_idx + 1}")

            batch_data = chunk[chunk['subject_id'].map(batch_lookup) == batch_num]
            
            if not batch_data.empty:
                collected_data.append(batch_data)
        
        # If we found data for this batch, clean and save it
        if collected_data:

            final_batch_data = pd.concat(collected_data, ignore_index=True)

            print_and_log_cleaning(final_batch_data)

            if cols_of_interest.get('code_version'):
                cleaned_batch_data = clean_data_batch_supportfunc(final_batch_data, cols_of_interest)
                cleaned_batch_data['coding_system'] = cols_of_interest['coding_system'] + cleaned_batch_data[cols_of_interest['code_version']]

                
            else:
                cleaned_batch_data = clean_data_batch_supportfunc(final_batch_data, cols_of_interest)
                cleaned_batch_data['coding_system'] = cols_of_interest['coding_system']

            cleaned_batch_data=cleaned_batch_data[['subject_id','date','code','coding_system']]
            
            # Prepare the output file path dynamically using the data_name and batch_num
            output_file = os.path.join(output_dir, f"{data_name.lower()}_batch{batch_num}_{filename}.csv")

            
            # Save the cleaned data to the file
            cleaned_batch_data.to_csv(output_file, index=False)

            clear_output(wait=True)
            display(cleaned_batch_data.head())
            print_and_log_cleaning(f"Batch {batch_num} cleaned data saved to {output_file}.")
        else:
            print_and_log_cleaning(f"Warning: No data found for batch {batch_num}.")
 
    
    
    display(f"\nCleaning complete. All batches processed and saved in the '{output_dir}' directory.")

## Handling Large Scale Data



Real world EHR datasets are typically too large to load into memory at once.  As a result, processing datasets in **batches** becomes crucial to speed up the data processing by processing data in a parallel fasion and ensuring you don't hit memory limits. 

For batching, the first step is to identify the **unique patients** since batching is usually performed at the patient level to make sure that all records for an individual are processed together. Once unique patients are identified, they can be split into batches for iterative or parallel processing.  




In [13]:
admissions_file  = os.path.join(base_directory, "raw_data","structured_data" ,"physionet.org", "files", "mimiciv", "3.1", "hosp","admissions.csv")

admissions = pd.read_csv(admissions_file, dtype=str)
display(admissions.head(5))

# Getting unique patient ids and sorting them
patient_ids = admissions[['subject_id']].drop_duplicates()
patient_ids = patient_ids.sort_values(by='subject_id', ascending=True)
patient_ids = patient_ids.reset_index(drop=True)
display(patient_ids.head())

# specify the number of batches you want to have. The larger the data, the more batches you need to have
num_of_batches = 8

# Assigning batch number from 1 to 8
patient_ids['batch_num'] = (patient_ids.index % num_of_batches) + 1
display(patient_ids)

patient_count_per_batch = patient_ids.groupby('batch_num')['subject_id'].count().reset_index().rename(columns={'subject_id': 'patient_count'})
display(patient_count_per_batch)

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


,subject_id
0,10000032
1,10000068
2,10000084
3,10000108
4,10000117


,subject_id,batch_num
0,10000032,1
1,10000068,2
2,10000084,3
3,10000108,4
4,10000117,5
...,...,...
223447,19999733,8
223448,19999784,1
223449,19999828,2
223450,19999840,3


,batch_num,patient_count
0,1,27932
1,2,27932
2,3,27932
3,4,27932
4,5,27931
5,6,27931
6,7,27931
7,8,27931


### Cleaning Diagnoses Data
Previously we did clean the diagnoses data, we are redoing the cleaning using the functions we defined.

In [14]:
diagnoses_icd_file = os.path.join(base_directory, "raw_data","structured_data", "physionet.org", "files", "mimiciv", "3.1", "hosp","diagnoses_icd.csv")
admissions_file  = os.path.join(base_directory, "raw_data", "structured_data" ,"physionet.org", "files", "mimiciv", "3.1", "hosp","admissions.csv")


diagnoses_icd = pd.read_csv(diagnoses_icd_file, dtype=str)
admissions = pd.read_csv(admissions_file, dtype=str)

timed_diagnoses_icd = pd.merge(
    diagnoses_icd,
    admissions[["subject_id", "hadm_id", "admittime"]],
    how="left",
    on=["subject_id", "hadm_id"],
)

display(timed_diagnoses_icd.head())

tmp_directory = os.path.join(base_directory, 'scripts', 'tmp')
print(f"Creating temp directory here {tmp_directory}")
os.makedirs(tmp_directory, exist_ok=True)

timed_diagnoses_icd_file = os.path.join(tmp_directory, f"timed_diagnoses_icd.csv")
timed_diagnoses_icd.to_csv(timed_diagnoses_icd_file, index=False)

,subject_id,hadm_id,seq_num,icd_code,icd_version,admittime
0,10000032,22595853,1,5723,9,2180-05-06 22:23:00
1,10000032,22595853,2,78959,9,2180-05-06 22:23:00
2,10000032,22595853,3,5715,9,2180-05-06 22:23:00
3,10000032,22595853,4,07070,9,2180-05-06 22:23:00
4,10000032,22595853,5,496,9,2180-05-06 22:23:00


Creating temp directory here /n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE/scripts/tmp


In [15]:
cols_of_interest = {
    "patient_id": "subject_id",
    "date": "admittime",
    "code": "icd_code",
    "code_version": "icd_version",
    "coding_system": "ICD"
}

clean_data_by_batch(
    timed_diagnoses_icd_file,
    cleaned_rawdata_directory,
    patient_ids,
    cols_of_interest,
    "Diagnoses",
    num_rows_to_load=15000000
)

,subject_id,date,code,coding_system
0,10000280,2151-03-18,6820,ICD9
1,10000886,2178-05-08,30500,ICD9
2,10001217,2157-11-18,3240,ICD9
3,10001217,2157-11-18,3484,ICD9
4,10001217,2157-11-18,3485,ICD9


Processing batches: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [00:35<00:00,  4.45s/batch]


"\nCleaning complete. All batches processed and saved in the '/n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE/processed_data/step3_cleaned_rawdata/Diagnoses' directory."

### Cleaning Procedures Data

In MIMIC Procedure data come from two sources: 
1. hcpcsevents.csv where procedures are recorded as CPT codes 
2. procedures_icd.csv where procedures are recorded as ICD9/ICD10 Procedure codes

We will need to clean them both and concatenate them.

In [16]:
# Cleaning HCPCS events
hcpcsevents_file_path = os.path.join(base_directory, "raw_data", "structured_data","physionet.org", "files", "mimiciv", "3.1", "hosp","hcpcsevents.csv")

hcpcsevents_sample = pd.read_csv(hcpcsevents_file_path ,nrows=5,dtype=str)
display(hcpcsevents_sample)

,subject_id,hadm_id,chartdate,hcpcs_cd,seq_num,short_description
0,10000068,25022803,2160-03-04,99218,1,Hospital observation services
1,10000084,29888819,2160-12-28,G0378,1,Hospital observation per hr
2,10000108,27250926,2163-09-27,99219,1,Hospital observation services
3,10000117,22927623,2181-11-15,43239,1,Digestive system
4,10000117,22927623,2181-11-15,G0378,2,Hospital observation per hr


In [17]:
cols_of_interest = {
    "patient_id": "subject_id",
    "date": "chartdate",
    "code": "hcpcs_cd",
    "code_version": None,
    "coding_system": "HCPCS"
}

clean_data_by_batch(
    hcpcsevents_file_path,
    cleaned_rawdata_directory,
    patient_ids,
    cols_of_interest,
    "Procedures",
    num_rows_to_load=15000000
)

,subject_id,date,code,coding_system
0,10000280,2151-03-18,99219,HCPCS
1,10000886,2178-05-08,99219,HCPCS
2,10002425,2153-01-07,27339,HCPCS
4,10002425,2153-01-07,G0378,HCPCS
5,10002807,2152-03-30,G0378,HCPCS


Processing batches: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [00:02<00:00,  3.87batch/s]


"\nCleaning complete. All batches processed and saved in the '/n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE/processed_data/step3_cleaned_rawdata/Procedures' directory."

In [18]:
# Cleaning procedures_icd.csv
procedures_icd_file = os.path.join(base_directory, "raw_data","structured_data", "physionet.org", "files", "mimiciv", "3.1", "hosp","procedures_icd.csv")
procedures_icd_sample = pd.read_csv(procedures_icd_file, dtype=str, nrows=5)
display(procedures_icd_sample)

,subject_id,hadm_id,seq_num,chartdate,icd_code,icd_version
0,10000032,22595853,1,2180-05-07,5491,9
1,10000032,22841357,1,2180-06-27,5491,9
2,10000032,25742920,1,2180-08-06,5491,9
3,10000068,25022803,1,2160-03-03,8938,9
4,10000117,27988844,1,2183-09-19,0QS734Z,10


In [19]:
cols_of_interest = {
    "patient_id": "subject_id",
    "date": "chartdate",
    "code": "icd_code",
    "code_version": "icd_version",
    "coding_system": "ICDPROC"
}

clean_data_by_batch(
    procedures_icd_file,
    cleaned_rawdata_directory,
    patient_ids,
    cols_of_interest,
    "Procedures",
    num_rows_to_load=15000000
)

,subject_id,date,code,coding_system
0,10000280,2151-03-18,8938,ICDPROC9
1,10000886,2178-05-08,8938,ICDPROC9
2,10001217,2157-11-20,0139,ICDPROC9
3,10001217,2157-11-19,0331,ICDPROC9
4,10001217,2157-11-22,3897,ICDPROC9


Processing batches: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [00:06<00:00,  1.26batch/s]


"\nCleaning complete. All batches processed and saved in the '/n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE/processed_data/step3_cleaned_rawdata/Procedures' directory."

### Cleaning Medications Data

In [20]:
prescriptions_file = os.path.join(base_directory, "raw_data", "structured_data", "physionet.org", "files", "mimiciv", "3.1", "hosp","prescriptions.csv")
prescriptions_sample = pd.read_csv(prescriptions_file,dtype=str,nrows=5)
display(prescriptions_sample)

,subject_id,hadm_id,pharmacy_id,poe_id,poe_seq,order_provider_id,starttime,stoptime,drug_type,drug,formulary_drug_cd,gsn,ndc,prod_strength,form_rx,dose_val_rx,dose_unit_rx,form_val_disp,form_unit_disp,doses_per_24_hrs,route
0,10000032,22595853,12775705,10000032-55,55,P85UQ1,2180-05-08 08:00:00,2180-05-07 22:00:00,MAIN,Furosemide,FURO40,008209,51079007320,40mg Tablet,NaN,40,mg,1,TAB,1,PO/NG
1,10000032,22595853,18415984,10000032-42,42,P23SJA,2180-05-07 02:00:00,2180-05-07 22:00:00,MAIN,Ipratropium Bromide Neb,IPRA2H,021700,00487980125,2.5mL Vial,NaN,1,NEB,1,VIAL,4,IH
2,10000032,22595853,23637373,10000032-35,35,P23SJA,2180-05-07 01:00:00,2180-05-07 09:00:00,MAIN,Furosemide,FURO20,008208,51079007220,20mg Tablet,NaN,20,mg,1,TAB,1,PO/NG
3,10000032,22595853,26862314,10000032-41,41,P23SJA,2180-05-07 01:00:00,2180-05-07 01:00:00,MAIN,Potassium Chloride,MICROK10,001275,00245004101,10mEq ER Tablet,NaN,40,mEq,4,TAB,1,PO
4,10000032,22595853,30740602,10000032-27,27,P23SJA,2180-05-07 00:00:00,2180-05-07 22:00:00,MAIN,Sodium Chloride 0.9% Flush,NACLFLUSH,NaN,0,10 mL Syringe,NaN,3,mL,0.3,SYR,3,IV


In [21]:
cols_of_interest = {
    "patient_id": "subject_id",
    "date": "starttime",
    "code": "ndc",
    "code_version": None,
    "coding_system": "NDC"
}

clean_data_by_batch(
    prescriptions_file,
    cleaned_rawdata_directory,
    patient_ids,
    cols_of_interest,
    "Medication",
    num_rows_to_load=15000000
)

,subject_id,date,code,coding_system
0,10001217,2157-11-22,08290036005,NDC
1,10001217,2157-11-21,00641608025,NDC
2,10001217,2157-11-22,00338001702,NDC
3,10001217,2157-11-22,00409433201,NDC
4,10001217,2157-11-21,0,NDC


Processing batches: 100%|███████████████████████████████████████████████████████████████████████| 8/8 [03:14<00:00, 24.28s/batch]


"\nCleaning complete. All batches processed and saved in the '/n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE/processed_data/step3_cleaned_rawdata/Medication' directory."

## Cleaning Lab Data: Handling Large-Scale Data

In [22]:
labitems_file = os.path.join(base_directory, "raw_data", "structured_data",  "physionet.org", "files", "mimiciv", "3.1", "hosp", "labevents.csv")
labs_sample= pd.read_csv(labitems_file,dtype=str,nrows=5)
display(labs_sample)

,labevent_id,subject_id,hadm_id,specimen_id,itemid,order_provider_id,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
0,1,10000032,NaN,2704548,50931,P69FQC,2180-03-23 11:51:00,2180-03-23 15:56:00,___,95,mg/dL,70,100,NaN,ROUTINE,"IF FASTING, 70-100 NORMAL, >125 PROVISIONAL DIABETES."
1,2,10000032,NaN,36092842,51071,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
2,3,10000032,NaN,36092842,51074,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN
3,4,10000032,NaN,36092842,51075,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,"BENZODIAZEPINE IMMUNOASSAY SCREEN DOES NOT DETECT SOME DRUGS,;INCLUDING LORAZEPAM, CLONAZEPAM, AND FLUNITRAZEPAM."
4,5,10000032,NaN,36092842,51079,P69FQC,2180-03-23 11:51:00,2180-03-23 16:00:00,NEG,NaN,NaN,NaN,NaN,NaN,ROUTINE,NaN


In [14]:
# Counting the number of lines in the lab file

file_line_count(labitems_file)

158374765

Lab data is one of the largest subdatasets in EHR due to the high frequency of recorded lab events. In MIMIC-IV, there are over 158 million lab observations recorded and loading the entire dataset into memory at once is inefficient. Hence, we will process the data in batches.

In [23]:
# This following will take atleast 3-4 minutes to process a single batch, and around 30 mins for the entire lab data

cols_of_interest = {
    "patient_id": "subject_id",
    "date": "charttime",
    "code": "itemid",
    "code_version": None,
    "coding_system": "ITEMID"
}

clean_data_by_batch(
    labitems_file,
    cleaned_rawdata_directory,
    patient_ids,
    cols_of_interest,
    "Labs",
    num_rows_to_load=15000000
)


# We process data in batches of patients, rather than batches of data, to efficiently handle duplicates, as we can't load the entire dataset into memory at once.
# If it takes too long to process a chunk, try reducing num_rows_to_load

,subject_id,date,code,coding_system
0,10000280,2151-03-18,51146,ITEMID
1,10000280,2151-03-18,51200,ITEMID
2,10000280,2151-03-18,51221,ITEMID
3,10000280,2151-03-18,51222,ITEMID
4,10000280,2151-03-18,51244,ITEMID


Processing batches: 100%|██████████████████████████████████████████████████████████████████████| 8/8 [16:58<00:00, 127.28s/batch]


"\nCleaning complete. All batches processed and saved in the '/n/data1/hsph/biostat/celehs/lab/va67/EHR_TUTORIAL_WORKSPACE/processed_data/step3_cleaned_rawdata/Labs' directory."

After running Step 3, you will have the following files generated under your workspace.

```bash
EHR_TUTORIAL_WORKSPACE/
└── processed_data/
    └── step3_cleaned_rawdata/
        ├── Diagnoses/
        │   ├── diagnoses_batch1_timed_diagnoses_icd.csv
        │   ├── diagnoses_batch2_timed_diagnoses_icd.csv
        │   ├── diagnoses_batch3_timed_diagnoses_icd.csv
        │   ├── diagnoses_batch4_timed_diagnoses_icd.csv
        │   ├── diagnoses_batch5_timed_diagnoses_icd.csv
        │   ├── diagnoses_batch6_timed_diagnoses_icd.csv
        │   ├── diagnoses_batch7_timed_diagnoses_icd.csv
        │   └── diagnoses_batch8_timed_diagnoses_icd.csv
        ├── Labs/
        │   ├── labs_batch1_labevents.csv
        │   ├── labs_batch2_labevents.csv
        │   ├── labs_batch3_labevents.csv
        │   ├── labs_batch4_labevents.csv
        │   ├── labs_batch5_labevents.csv
        │   ├── labs_batch6_labevents.csv
        │   ├── labs_batch7_labevents.csv
        │   └── labs_batch8_labevents.csv
        ├── Medication/
        │   ├── medication_batch1_prescriptions.csv
        │   ├── medication_batch2_prescriptions.csv
        │   ├── medication_batch3_prescriptions.csv
        │   ├── medication_batch4_prescriptions.csv
        │   ├── medication_batch5_prescriptions.csv
        │   ├── medication_batch6_prescriptions.csv
        │   ├── medication_batch7_prescriptions.csv
        │   └── medication_batch8_prescriptions.csv
        └── Procedures/
            ├── procedures_batch1_hcpcsevents.csv
            ├── procedures_batch2_hcpcsevents.csv
            ├── procedures_batch3_hcpcsevents.csv
            ├── procedures_batch4_hcpcsevents.csv
            ├── procedures_batch5_hcpcsevents.csv
            ├── procedures_batch6_hcpcsevents.csv
            ├── procedures_batch7_hcpcsevents.csv
            └── procedures_batch8_hcpcsevents.csv

```